In [0]:
# CAPA TRUSTED — limpieza, validacion y enriquecimiento
# recibimos 3M de viajes crudos y entregamos solo los validos

import time
from pyspark.sql import functions as f

CATALOGO     = "nyc_taxi_andres"
CAPA_RAW     = f"{CATALOGO}.raw"
CAPA_TRUSTED = f"{CATALOGO}.trusted"

print("Configuracion lista, cargando datos desde Raw...")

In [0]:
# leemos las tablas que dejamos en la capa raw
viajes = spark.table(f"{CAPA_RAW}.viajes_enero_2023")
zonas  = spark.table(f"{CAPA_RAW}.zonas_taxi")

total_inicial = viajes.count()
print(f"Viajes que entran a Trusted: {total_inicial:,}")

In [0]:
# aplicamos las reglas de calidad una por una
# guardamos cuantos registros descarta cada regla
# eso nos sirve para el reporte final y el KPI de calidad


descartados = {}  # aqui vamos acumulando los descartados por regla

# regla 1: el viaje debe tener un tiempo valido (pickup < dropoff).
# si el taxi llega antes de salir, algo estuvo mal en el sistema
sin_tiempo_valido = viajes.filter(
    f.col("tpep_dropoff_datetime") <= f.col("tpep_pickup_datetime")
).count()
descartados["tiempo_invalido"] = sin_tiempo_valido
viajes = viajes.filter(
    f.col("tpep_dropoff_datetime") > f.col("tpep_pickup_datetime")
)
print(f"Regla 1 - tiempo invalido: {sin_tiempo_valido:,} descartados")


# regla 2: el viaje debe tener distancia positiva. (trip_distance > 0).
# un viaje sin distancia no es un viaje
sin_distancia = viajes.filter(
    f.col("trip_distance") <= 0
).count()
descartados["sin_distancia"] = sin_distancia
viajes = viajes.filter(f.col("trip_distance") > 0)
print(f"Regla 2 - sin distancia: {sin_distancia:,} descartados")


# regla 3: la tarifa debe ser positiva (fare_amount > 0).
# tarifas negativas o en cero son errores del sistema de cobro
tarifa_invalida = viajes.filter(
    f.col("fare_amount") <= 0
).count()
descartados["tarifa_invalida"] = tarifa_invalida
viajes = viajes.filter(f.col("fare_amount") > 0)
print(f"Regla 3 - tarifa invalida: {tarifa_invalida:,} descartados")


# regla 4: descartamos outliers extremos.
# viajes de mas de 4 horas son raros para taxis amarillos en NYC
# pueden ser errores de cierre de viaje en el sistema
# el valor de $300 es mas normal para los viajes al aeropuerto que pueden ser mas costosos.
viajes = viajes.withColumn(
    "duracion_horas",
    (f.unix_timestamp("tpep_dropoff_datetime") - 
     f.unix_timestamp("tpep_pickup_datetime")) / 3600
)
outliers = viajes.filter(
    (f.col("duracion_horas") > 4) |
    (f.col("fare_amount") > 300)
).count()
descartados["outliers_extremos"] = outliers
viajes = viajes.filter(
    (f.col("duracion_horas") <= 4) &
    (f.col("fare_amount") <= 300)
)
print(f"Regla 4 - outliers extremos: {outliers:,} descartados")


# regla 5: descartamos nulos en columnas clave.
nulos = viajes.filter(
    f.col("PULocationID").isNull() |
    f.col("tpep_pickup_datetime").isNull() |
    f.col("fare_amount").isNull()
).count()
descartados["nulos_columnas_clave"] = nulos
viajes = viajes.filter(
    f.col("PULocationID").isNotNull() &
    f.col("tpep_pickup_datetime").isNotNull() &
    f.col("fare_amount").isNotNull()
)
print(f"Regla 5 - nulos en columnas clave: {nulos:,} descartados")

# resumen de la limpieza
total_descartados = sum(descartados.values())
total_limpio = viajes.count()
porcentaje_descartado = round((total_descartados / total_inicial) * 100, 2)

print(f"\nTotal descartados : {total_descartados:,} ({porcentaje_descartado}%)")
print(f"Total que quedan  : {total_limpio:,}")

In [0]:
# Join con Taxi Zones.(cruzamos por PULocationID y traemos Borough y Zone)
# cruzamos cada viaje con la tabla de zonas para saber
# en que barrio empezo el viaje, no solo el ID numerico

print("Enriqueciendo viajes con informacion de zonas...")

# renombramos las columnas del lookup para que no choquen
zonas_pickup = zonas.select(
    f.col("LocationID").alias("zona_id"),
    f.col("Borough").alias("barrio_origen"),
    f.col("Zone").alias("zona_origen"),
    f.col("service_zone").alias("tipo_zona_origen")
)

# hacemos el join por el ID de zona de pickup
viajes = viajes.join(
    zonas_pickup,
    viajes["PULocationID"] == zonas_pickup["zona_id"],
    how="left"
)

# cuantos viajes quedaron sin zona reconocida
sin_zona = viajes.filter(f.col("barrio_origen").isNull()).count()
print(f"Viajes sin zona reconocida: {sin_zona:,}")
print("Enriquecimiento completado")

In [0]:
# estaandarizacion de nombre de columnas.
# renombramos las columnas al español
# y nos quedamos solo con las que tienen valor para el negocio

viajes_trusted = viajes.select(
    f.col("tpep_pickup_datetime").alias("fecha_hora_inicio"),
    f.col("tpep_dropoff_datetime").alias("fecha_hora_fin"),
    f.col("duracion_horas").alias("duracion_horas"),
    f.col("passenger_count").alias("cantidad_pasajeros"),
    f.col("trip_distance").alias("distancia_millas"),
    f.col("PULocationID").alias("id_zona_origen"),
    f.col("DOLocationID").alias("id_zona_destino"),
    f.col("fare_amount").alias("tarifa_base"),
    f.col("tip_amount").alias("propina"),
    f.col("tolls_amount").alias("peajes"),
    f.col("total_amount").alias("total_cobrado"),
    f.col("payment_type").alias("tipo_pago"),
    f.col("barrio_origen"),
    f.col("zona_origen"),
    f.col("tipo_zona_origen")
)

print(f"Columnas finales: {len(viajes_trusted.columns)}")
print(viajes_trusted.columns)

In [0]:
# guardamos la tabla limpia y enriquecida en la capa trusted
print("Guardando en la capa Trusted...")
inicio = time.time()

try:
    (viajes_trusted
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{CAPA_TRUSTED}.viajes_limpios")
    )
    print(f"Tabla guardada en {round(time.time() - inicio, 2)}s")
except Exception as e:
    print(f"Error guardando en Trusted: {e}")
    raise

In [0]:
# resumen final de la capa trusted

print("  RESUMEN — CAPA TRUSTED")
print("=" * 55)
print(f"  Viajes que entraron  : {total_inicial:,}")
print(f"  Viajes descartados   : {total_descartados:,} ({porcentaje_descartado}%)")
print(f"  Viajes que quedaron  : {total_limpio:,}")
print(f"")
print(f"  Detalle de descartados:")
for regla, cantidad in descartados.items():
    pct = round((cantidad / total_inicial) * 100, 2)
    print(f"    {regla}: {cantidad:,} ({pct}%)")
print("=" * 55)
print("  Siguiente paso: KPIs en Refined")
